In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F
from pyspark.sql.functions import col, datediff, current_date, sum, countDistinct

#silver table filepaths
silver_table_customers = "dev.trailsport.silver_customers"
silver_table_products = "dev.trailsport.silver_products"
silver_table_sales = "dev.trailsport.silver_sales"
silver_table_sales_items = "dev.trailsport.silver_sales_items"

#gold table filepath
gold_table_analytics = "dev.trailsport.gold_analytics"
gold_table_science = "dev.trailsport.gold_science"

# 1 Load silver data and write to Gold for analytics

In [0]:
# Pull the differen tables in pd df
df_customers = spark.read.table(silver_table_customers)
df_products = spark.read.table(silver_table_products)
df_sales = spark.read.table(silver_table_sales)
df_sales_items = spark.read.table(silver_table_sales_items)

# Force all join keys to be strings so PySpark doesn't try to cast them to BIGINTs during the join
df_sales_items = df_sales_items.withColumn("SaleID", col("SaleID").cast("string"))
df_sales = df_sales.withColumn("SaleID", col("SaleID").cast("string")).withColumn("CustomerID", col("CustomerID").cast("string"))
df_customers = df_customers.withColumn("CustomerID", col("CustomerID").cast("string"))

### 1.1 Create calculated dataframes

In [0]:
# create age and customer since based on difference between date and today
df_customers = df_customers.withColumn("Age",(datediff(current_date(),col("Birthdate"))/365).cast("integer"))\
    .withColumn("Days_is_Customer", datediff(current_date(),col("CustomerSince")))

#create count of products per sale
df_count = (df_sales_items.groupBy("SaleID", "ProductID").count()
    .withColumnRenamed("count", "Order_Quantity"))

# window to look at the order
window_sale = Window.partitionBy("SaleID")

#window to look total product sales
window_product = Window.partitionBy("ProductID")

df_prep_spending = (
    df_count
    .join(df_products, on="ProductID", how="left").dropna(subset=["Price"])
    .join(df_sales, on="SaleID", how="left")
    .withColumn("Item_Revenue", col("Order_Quantity") * col("Price"))
    .withColumn("SalePrice_calc", sum("Item_Revenue").over(window_sale))
    .withColumn("Total_Lifetime_Product_Units", sum("Order_Quantity").over(window_product))
)

#Calculate spending per customer. This will be in the gold table.
df_spending = df_prep_spending.groupBy("CustomerID").agg(
    sum("Item_Revenue").alias("Total_Amount_Spent"), # Sums up the math we did above
    countDistinct("SaleID").alias("Total_Orders_Customer"))


### 1.2 Join the dataframes

In [0]:
# Join all tables together to create the One Big Table (OBT)
df_obt_raw = (
    df_prep_spending 
    .join(df_customers, on="CustomerID", how="inner") 
    .join(df_spending, on="CustomerID", how="left")  
)

### 1.3 Select columns from dataframes

In [0]:
df_gold_bi = (
    df_obt_raw    
    # Select only what the BI analyst needs, and rename for clarity
    .select(
        # Sale info
        col("SaleID"),
        col("SaleDate"), 
        col("SalePrice"),
        col("SalePrice_calc").cast("double"),
        # Customer info (Keeping the readable text, NOT the ML encoded 1s and 0s!)
        col("CustomerID").cast("integer"),
        col("CustomerName"),
        col("Age"),
        col("Gender"),
        col("OwnsHouse"),
        col("OwnsCar"),
        col("Days_is_Customer"),
        
        # Product info
        col("ProductID"),
        col("Product"),
        col("Price").cast("double"),
        col("ProductCategory"),
        col("ProductType"),
        
        # Metrics
        col("Total_Orders_Customer"),
        col("Total_Lifetime_Product_Units"),
        col("Order_Quantity"))
    .orderBy("SaleID")
)

In [0]:
df_gold_bi.limit(10).display()

SaleID,SaleDate,SalePrice,SalePrice_calc,CustomerID,CustomerName,Age,Gender,OwnsHouse,OwnsCar,Days_is_Customer,ProductID,Product,Price,ProductCategory,ProductType,Total_Orders_Customer,Total_Lifetime_Product_Units,Order_Quantity
100007,2023-11-19,null,65.99,81017,B.T. Wolff,31,Male,false,false,2245,454924,null,65.99,Footwear,Boots,2,2428,1
100008,2023-10-14,65.99,65.99,64613,D.F. Jansen,null,Female,true,true,1980,454924,null,65.99,Footwear,Boots,2,2428,1
100009,2022-09-21,null,2.29,37462,O. Stoot,28,Male,false,true,1367,435934,Patch kit,2.29,Accessories,Tires and Tubes,2,17590,1
10001,2022-02-07,211.8,211.8,86411,R. Franken,29,Female,true,true,2528,410104,Course Pro Golf Bag,206.9,Golf Equipment,Golf Accessories,3,4427,1
10001,2022-02-07,211.8,211.8,86411,R. Franken,29,Female,true,true,2528,452752,null,4.9,null,Sunscreen,3,4425,1
100010,2022-04-04,null,139.99,69803,C. Commissaris,45,Female,false,true,1497,443394,Football Copa 19.1 Firm Ground Cleats,139.99,Footwear,Cleats,2,1778,1
100014,2020-04-14,null,147.29,27161,V. Montulet,30,Male,false,false,3132,425461,Astro Pilot,145.0,Personal Accessories,Navigation,2,2830,1
100014,2020-04-14,null,147.29,27161,V. Montulet,30,Male,false,false,3132,435934,Patch kit,2.29,Accessories,Tires and Tubes,2,17590,1
100017,2020-06-30,null,87.68,34404,K.M. Walstra,31,Female,false,true,2654,426367,Bear Survival Edge,87.68,Personal Accessories,Knives,1,1356,1
100022,2022-10-10,null,202.53,20028,R.S.E. Kager,null,Male,null,null,1291,453049,Course Pro Golf and Tee Set,10.24,Golf Equipment,Golf Accessories,4,2507,1


### 1.4 Write into gold table

In [0]:

# Write it to the Gold Layer as a Delta Table!
(df_gold_bi.write
    .format("delta")
    .option("overwriteSchema", "true")
    .mode("overwrite") 
    .saveAsTable(gold_table_analytics)
)